# 第9周 · 周四：PSM——为每个"处理过的人"找一个"双胞胎"\n\n> 难度：⭐⭐⭐⭐ | 定位：理解PSM的原理、步骤和局限，能用Python实现完整的PSM分析\n> 前置依赖：周一因果推断基础、逻辑回归 | 后续用途：周五PSM平衡性检验、W10-W12综合案例\n> 面试地位：观测因果推断中最常用的方法，面试高频
\n---\n\n## 📘 区块1：核心概念与第一性原理\n\n### 1.1 从"条件期望"到"倾向得分"——PSM的数学直觉\n\nPSM解决的核心问题：当处理不是随机分配的，如何比较"可比的"处理组和对照组？\n\n假设你想知道"上大学"对收入的影响。直接比较"上过大学的"和"没上过大学的"不公平——上过大学的人可能本来就聪明/努力/家境好。即使不上大学，他们收入也可能更高。\n\n**理想方案**：对于每个上过大学的人，找一个"没上大学但其他方面（智力、努力程度、家境）几乎一模一样"的人。对比这两个人的收入→就是上大学的因果效应。\n\n**PSM的方案**：不用在所有维度上逐项匹配（维度一多就找不到人了——"维度诅咒"），而是先估算每个人"被处理的概率"（倾向得分），然后匹配这个单一数字。\n\n**为什么倾向得分有效**：Rosenbaum和Rubin(1983)证明了——如果给定协变量X，处理分配是可忽略的(即(Y(0),Y(1)) ⟂ T | X)，那么在给定倾向得分p(X)的条件下，处理分配也是可忽略的。\n\n这意味着：**匹配倾向得分等价于匹配所有协变量**——前提是倾向得分模型正确指定了。\n\n### 1.2 PSM的四个步骤\n\n```\n步骤1：估计倾向得分\n      → 用logistic回归(或其他分类器)建模"被处理的概率"\n      → 因变量：是否被处理(0/1)\n      → 自变量：所有可能的混淆变量\n      → 输出：每个人一个[0,1]之间的倾向得分\n\n步骤2：匹配\n      → 对每个处理组个体，找一个倾向得分最接近的对照组个体\n      → 多种匹配方法：最近邻/半径/核匹配/分层匹配\n      → 选项：有放回 vs 无放回，1对1 vs 1对多\n\n步骤3：诊断匹配质量（平衡性检验）\n      → 匹配后，各协变量在两组之间的SMD应<0.1\n      → 检查倾向得分的分布是否有足够的重叠区域\n      → 没通过检验→回去调步骤1或步骤2\n\n步骤4：估计处理效应\n      → ATT = 匹配后处理组均值 - 匹配后对照组均值\n      → （通常估计ATT，不是ATE——因为PSM对处理组的匹配质量通常更好）\n```\n\n### 1.3 PSM的识别假设——你需要相信什么\n\nPSM依赖两个核心假设：\n\n**假设1：可忽略性(Unconfoundedness)**\n在给定协变量X的条件下，处理分配与潜在结果独立。\n白话：你测量到的X包含了所有同时影响"是否被处理"和"结果"的变量。没有遗漏重要的混淆变量。\n\n**假设2：共同支撑(Common Support / Overlap)**\n对于X的任何取值，个体被处理和不被处理的概率都严格在(0,1)之间。\n白话：处理组和对照组的人要有"可比性"——不能处理组所有人都100%会被处理（倾向得分都接近1），而对照组没有人会（倾向得分都接近0）。\n\n**假设1是不可检验的**——这就是PSM最大的软肋。如果有未观测的混淆变量（如"动机"、"天赋"），PSM仍然有偏。\n\n### 1.4 面试高频考点\n\n**面试官常问**：\n- "PSM的四个步骤是什么？"\n- "为什么倾向得分能降维——在X上匹配和在p(X)上匹配有什么区别？"\n- "PSM和DID有什么区别？什么时候用哪个？"\n- "PSM最大的局限是什么？"\n\n**回答要点**：\n- 估计倾向得分→匹配→诊断→估计效应\n- p(X)将多维X压缩为一个标量；在p(X)上匹配避免了"维度诅咒"\n- DID利用时间维度的变化(需要前后+对照+平行趋势)，PSM利用横截面的相似性(需要丰富的协变量)\n- PSM最大局限：只控制了可观测的混淆，未观测混淆导致偏误无法消除

## 🧩 区块2：知识点详细拆解\n\n### 2.1 倾向得分的五种估计方法\n\n**为什么重要**：不同方法可能给出不同的倾向得分，进而影响匹配质量和最终结论。\n\n| 方法 | 优点 | 缺点 | 适用场景 |\n|------|------|------|----------|\n| **Logistic回归** | 简单、可解释、标准误可用 | 假设线性(在logit尺度上) | 最常用，基准方案 |\n| **Probit回归** | 类似logistic，正态假设 | 与logistic差异不大 | 某些学科偏好 |\n| **GBDT** | 自动捕捉非线性+交互 | 过拟合风险，统计推断困难 | 变量多+关系复杂 |\n| **LASSO logistic** | 自动变量选择+正则化 | 可能漏掉弱混淆变量 | 高维数据(p>>n) |\n| **因果森林** | 专为因果推断设计 | 较新，实现较少 | 异质性处理效应 |\n\n**选择建议**：\n- 起步用Logistic回归——它是基准，如果Logistic做不好再升级\n- 变量很多(>50)且关系复杂→考虑GBDT或LASSO\n- 做学术研究→倾向得分模型的选择要做敏感性分析（不同模型→结论是否一致？）\n\n### 2.2 五种匹配算法对比\n\n**为什么重要**：匹配方法的选择直接影响匹配的样本量、平衡性和估计精度。\n\n| 匹配方法 | 原理 | 优点 | 缺点 |\n|---------|------|------|------|\n| **最近邻(1:1)** | 对每个处理组，找1个ps最近的对照 | 简单直观 | 可能匹配到"不太像"的人 |\n| **最近邻(1:K)** | 对每个处理组，找K个ps最近的对照 | 降低方差(更多样本) | 可能引入偏误(匹配质量下降) |\n| **卡尺匹配** | 只在ps差距<caliper的范围内匹配 | 保证匹配质量 | 部分处理组可能找不到匹配 |\n| **半径匹配** | 用ps差距<radius的所有对照加权平均 | 使用更多信息 | 需要指定radius |\n| **核匹配** | 用所有对照组加权(权重=核函数(ps差距)) | 使用全部信息 | 边界效应差 |\n| **分层匹配** | 按ps分位数分层,层内直接比较 | 稳健 | 粗粒度,可能不精细 |\n\n**推荐实践**：\n- 首选：最近邻(1:1)+卡尺(caliper=0.2×SD(ps))+无放回\n- 如果匹配掉了太多样本→考虑1:K(K=3或5)或半径匹配\n- 报告中展示多种匹配方法的结果(敏感性分析)\n\n### 2.3 卡尺(Caliper)的选择\n\n**为什么重要**：卡尺是匹配质量的"守门人"——太宽(匹配到不像的人)、太窄(找不到匹配)。\n\n**常用标准**：caliper = 0.2 × SD(倾向得分)\n- 这是Austin(2011)通过大量模拟研究推荐的标准\n- 在这个卡尺下，约能消除98-99%的倾向得分偏差\n\n**实践中的权衡**：\n- 如果你用caliper=0.2×SD但损失了>20%的处理组→考虑放宽到0.3或0.5\n- 但一定要报告："在用0.2×SD卡尺时损失了X%样本，用0.5×SD时结论是否改变"\n\n### 2.4 有放回 vs 无放回匹配\n\n**为什么重要**：两种方式在偏误-方差权衡上有本质区别。\n\n| 特性 | 有放回 | 无放回 |\n|------|-------|--------|\n| 匹配质量 | 更好（每次选最优） | 可能差一些（最优的已被用掉）|\n| 估计方差 | 更大（对照组样本被重复使用） | 更小 |\n| 偏误 | 更小 | 可能更大 |\n| 样本利用率 | 可能只用少数"好"对照组 | 对照组利用更充分 |\n| 标准误 | 需要特殊处理 | 相对标准 |\n\n**选择**：偏误优先→有放回；方差优先→无放回。学术研究中更多用有放回+Abadie-Imbens标准误。\n\n### 2.5 PSM+DID组合——取两家之长\n\n**为什么重要**：这是实证研究中最常用的"增强版"因果推断策略。\n\n**做法**：\n1. 用处理前的协变量做PSM（只能用处理前的！不能用处理后的）\n2. 在匹配后的样本上做DID\n3. DID消除了不随时间变化的未观测混淆(固定效应)\n4. PSM让处理组和对照组在处理前更可比\n\n**优势**：结合了PSM（控制可观测混淆）和DID（控制不可观测的不变混淆）的优势。\n\n**注意事项**：不能用处理后的协变量做PSM——这会导致"控制未来"或"控制中介变量"的偏误。\n\n### 2.6 倾向得分加权(IPTW)——不匹配，改为加权\n\n**为什么重要**：IPTW是PSM的替代方案，避免了"匹配丢失样本"的问题。\n\n**原理**：\n- 对处理组个体赋予权重 = 1/p(X)\n- 对对照组个体赋予权重 = 1/(1-p(X))\n- 加权后的样本相当于"假想的随机化样本"\n\n**优势**：不丢失样本\n**劣势**：对倾向得分模型极敏感——极端倾向得分(接近0或1)会导致极大权重→方差爆炸→需要权重修剪(trimming)或稳定权重\n\n### 2.7 倾向得分分位数分层——最朴素的PSM\n\n**为什么重要**：当你不确定倾向得分模型是否正确时，分层法是最稳健的选择。\n\n**做法**：\n1. 估计倾向得分\n2. 按倾向得分分位数分成5-10层\n3. 在每一层内直接比较处理组和对照组的均值\n4. 总体效应 = 各层效应的加权平均\n\n**为什么5层就够了**：Cochran(1968)证明，在连续变量的5个分位数层内，约能消除90%的偏差。这是因为分位数分层是一个很好的"非参数"匹配方式。

## 💻 区块3：动手练习\n\n### 步骤0：环境准备

In [ ]:
import numpy as np; import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import NearestNeighbors
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('='*60)
print('PSM倾向得分匹配——动手练习')
print('='*60)

### 步骤1：基础练习——四种估计方法对比

1.1 朴素对比(有偏)→回归调整→PSM最近邻→PSM卡尺匹配
1.2 对比四种方法的偏误大小

In [ ]:
# 步骤1.1：生成带选择偏差的数据
print('【步骤1.1】生成数据：上大学对收入的影响')
print('-'*50)

n = 2000
# 可观测协变量
ability = np.random.normal(0, 1, n)      # 能力
family = np.random.normal(0, 1, n)       # 家境
# 未观测混淆（现实中你看不到！我们在这里模拟它）
motivation = np.random.normal(0, 1, n)   # 动机（未观测）

# 处理分配：上大学与否——跟能力、家境、动机都有关
prob_college = 1 / (1 + np.exp(-(-0.5 + 0.8*ability + 0.6*family + 0.5*motivation)))
college = np.random.binomial(1, prob_college)

# 收入：跟能力、家境、动机都有关，上大学真实效应=8000元/年
true_effect = 8000
income = (30000 + 5000*ability + 3000*family + 4000*motivation 
          + true_effect*college + np.random.normal(0, 5000, n))

df = pd.DataFrame({'ability': ability, 'family': family, 
                   'college': college, 'income': income, 'motivation': motivation})

print(f'处理组(上过大学): {(college==1).sum()}人')
print(f'对照组(没上大学): {(college==0).sum()}人')
print(f'真实因果效应: {true_effect}元/年')
print()

# --- 方法1：朴素对比 ---
naive = df[df.college==1]['income'].mean() - df[df.college==0]['income'].mean()
print(f'方法1-朴素对比: {naive:.0f}元 (偏误: {naive-true_effect:.0f})')

# --- 方法2：OLS回归（控制能力+家境）---
X_reg = df[['ability', 'family', 'college']]
reg = LinearRegression().fit(X_reg, df['income'])
reg_est = reg.coef_[2]
print(f'方法2-OLS回归: {reg_est:.0f}元 (偏误: {reg_est-true_effect:.0f}) ← 仍有偏(未控制motivation)')

# --- 方法3：PSM最近邻匹配 ---
lr = LogisticRegression()
lr.fit(df[['ability', 'family']], df['college'])
df['ps'] = lr.predict_proba(df[['ability', 'family']])[:, 1]

# 最近邻匹配（无放回，1:1）
treated_df = df[df.college==1].copy()
control_df = df[df.college==0].copy()
matched_incomes = []
for _, row in treated_df.iterrows():
    idx = (control_df['ps'] - row['ps']).abs().idxmin()
    matched_incomes.append(control_df.loc[idx, 'income'])
    control_df = control_df.drop(idx)

psm_nn = treated_df['income'].mean() - np.mean(matched_incomes)
print(f'方法3-PSM最近邻: {psm_nn:.0f}元 (偏误: {psm_nn-true_effect:.0f})')

# --- 方法4：PSM卡尺匹配(caliper=0.2*SD(ps)) ---
caliper = 0.2 * df['ps'].std()
print(f'\\n卡尺: {caliper:.4f}')

control_df2 = df[df.college==0].copy()
matched_incomes2 = []
n_lost = 0
for _, row in treated_df.iterrows():
    distances = (control_df2['ps'] - row['ps']).abs()
    if distances.min() <= caliper:
        idx = distances.idxmin()
        matched_incomes2.append(control_df2.loc[idx, 'income'])
        control_df2 = control_df2.drop(idx)
    else:
        n_lost += 1

psm_caliper = treated_df['income'].mean() - np.mean(matched_incomes2)
print(f'方法4-PSM卡尺: {psm_caliper:.0f}元 (偏误: {psm_caliper-true_effect:.0f}, 丢失{n_lost}人)')

print()
print('注意：PSM仍有些许偏误——因为motivation(动机)未观测+未被纳入倾向得分模型')
print('这就是PSM的根本局限：只控制了可观测的混淆。')

### 步骤2：进阶练习——匹配算法的深入对比

2.1 可视化倾向得分分布（匹配前 vs 匹配后）
2.2 对比不同K(1:1/1:3/1:5)的匹配效果

In [ ]:
# 步骤2.1：倾向得分分布——匹配前 vs 匹配后
print('\n【步骤2.1】倾向得分分布对比')
print('-'*50)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 匹配前：全样本
for label, color in [('处理组', 'red'), ('对照组', 'blue')]:
    sub = df[df.college==(1 if label=='处理组' else 0)]
    axes[0].hist(sub['ps'], bins=30, alpha=0.5, color=color, label=label, density=True)
axes[0].set_title('匹配前倾向得分分布')
axes[0].set_xlabel('倾向得分'); axes[0].legend()

# 匹配后
matched_ps = df.loc[control_df.index.intersection(df[df.college==0].index), 'ps']
axes[1].hist(treated_df['ps'], bins=30, alpha=0.5, color='red', label='处理组', density=True)
axes[1].hist(matched_ps, bins=30, alpha=0.5, color='blue', label='匹配上的对照组', density=True)
axes[1].set_title('匹配后倾向得分分布')
axes[1].set_xlabel('倾向得分'); axes[1].legend()

# 共同支撑区域（高亮重叠部分）
axes[2].hist(df[df.college==1]['ps'], bins=30, alpha=0.5, color='red', label='处理组', density=True)
axes[2].hist(df[df.college==0]['ps'], bins=30, alpha=0.5, color='blue', label='对照组(全部)', density=True)
overlap_min = max(df[df.college==1]['ps'].min(), df[df.college==0]['ps'].min())
overlap_max = min(df[df.college==1]['ps'].max(), df[df.college==0]['ps'].max())
axes[2].axvspan(overlap_min, overlap_max, alpha=0.15, color='green', label='共同支撑区域')
axes[2].set_title('共同支撑(Overlap)检查')
axes[2].set_xlabel('倾向得分'); axes[2].legend()
plt.tight_layout(); plt.show()

print(f'共同支撑区域: [{overlap_min:.3f}, {overlap_max:.3f}]')
off_support = ((df['ps'] < overlap_min) | (df['ps'] > overlap_max)).sum()
print(f'不在共同支撑区域的样本: {off_support}人 ({off_support/n*100:.1f}%)')
if off_support / n < 0.1:
    print('→ 共同支撑良好 ✓')
else:
    print('→ 超过10%样本不在共同支撑区域 ⚠ 需要修剪(trimming)')

In [ ]:
# 步骤2.2：不同K的最近邻匹配效果对比
print('\n【步骤2.2】1:1 vs 1:K匹配效果对比')
print('-'*50)

from sklearn.neighbors import NearestNeighbors

control_ps = df[df.college==0][['ps']].values
treated_ps = df[df.college==1][['ps']].values

for K in [1, 3, 5]:
    nn = NearestNeighbors(n_neighbors=K)
    nn.fit(control_ps)
    distances, indices = nn.kneighbors(treated_ps)
    
    # 对每个处理组，取K个匹配对照的均值
    control_income = df[df.college==0]['income'].values
    matched_income_mean = np.array([control_income[idx].mean() for idx in indices])
    att_k = df[df.college==1]['income'].values.mean() - matched_income_mean.mean()
    avg_dist = distances.mean()
    
    print(f'  1:{K}匹配: ATT={att_k:.0f}元, 平均ps距离={avg_dist:.5f}')

print()
print('解读：K越大→方差越小，但ps距离越大→偏误可能越大')
print('     需要在偏误(ps距离)和方差(K大小)之间权衡')

### 步骤3：挑战练习——PSM的完整分析闭环

3.1 模拟一个真实商业场景（VIP会员的效果评估）
3.2 倾向得分分层+匹配+结果解释

In [ ]:
# 步骤3：完整PSM分析——VIP会员的增量消费
print('\n【步骤3】完整PSM：VIP会员对用户消费的因果效应')
print('='*60)

np.random.seed(2024)
n = 5000

# 用户特征
age = np.random.uniform(18, 60, n)
tenure = np.random.exponential(12, n)      # 注册月数
historical_spend = np.random.lognormal(5, 1, n)  # 历史消费
active_days = np.random.poisson(15, n)     # 月活跃天数
unobserved_loyalty = np.random.normal(0, 1, n)  # 未观测的忠诚度

# VIP开通概率：跟可观测特征+不可观测忠诚度相关
prob_vip = 1 / (1 + np.exp(-(-1.5 + 0.02*(age-35) + 0.05*tenure + 
    0.0003*historical_spend + 0.03*active_days + 0.8*unobserved_loyalty)))
is_vip = np.random.binomial(1, prob_vip)

# 月消费：真实VIP效应=200元/月
true_vip_effect = 200
monthly_spend = (500 + 2*age + 10*tenure + 0.3*historical_spend 
                 + 30*active_days + 100*unobserved_loyalty
                 + true_vip_effect*is_vip + np.random.normal(0, 200, n))

df_vip = pd.DataFrame({
    'age': age, 'tenure': tenure, 'historical_spend': historical_spend,
    'active_days': active_days, 'is_vip': is_vip, 'monthly_spend': monthly_spend,
    'loyalty': unobserved_loyalty
})

print(f'VIP用户: {is_vip.sum()}人 ({is_vip.sum()/n*100:.1f}%)')
print(f'非VIP用户: {(1-is_vip).sum()}人')
print()

# 1. 朴素对比
naive_vip = df_vip[df_vip.is_vip==1]['monthly_spend'].mean() - df_vip[df_vip.is_vip==0]['monthly_spend'].mean()
print(f'1. 朴素对比: {naive_vip:.0f}元/月 (真实={true_vip_effect})')

# 2. OLS回归
X_vip = df_vip[['age', 'tenure', 'historical_spend', 'active_days', 'is_vip']]
reg_vip = LinearRegression().fit(X_vip, df_vip['monthly_spend'])
ols_vip = reg_vip.coef_[4]
print(f'2. OLS回归: {ols_vip:.0f}元/月 (真实={true_vip_effect})')

# 3. PSM
lr_vip = LogisticRegression(max_iter=1000)
lr_vip.fit(df_vip[['age', 'tenure', 'historical_spend', 'active_days']], df_vip['is_vip'])
df_vip['ps_vip'] = lr_vip.predict_proba(df_vip[['age', 'tenure', 'historical_spend', 'active_days']])[:, 1]

# 卡尺匹配
caliper_vip = 0.2 * df_vip['ps_vip'].std()
treated_vip = df_vip[df_vip.is_vip==1].copy()
control_vip = df_vip[df_vip.is_vip==0].copy()
matched_spends = []
n_lost = 0
for _, row in treated_vip.iterrows():
    distances = (control_vip['ps_vip'] - row['ps_vip']).abs()
    if distances.min() <= caliper_vip:
        idx = distances.idxmin()
        matched_spends.append(control_vip.loc[idx, 'monthly_spend'])
        control_vip = control_vip.drop(idx)
    else:
        n_lost += 1

psm_vip = treated_vip['monthly_spend'].mean() - np.mean(matched_spends)
print(f'3. PSM(卡尺): {psm_vip:.0f}元/月 (真实={true_vip_effect}, 丢失{n_lost}人)')
print()

# 汇总对比
print('='*60)
print(f'方法对比 (真实效应={true_vip_effect}元/月):')
print(f'  朴素对比: {naive_vip:>8.0f} (偏误 {naive_vip-true_vip_effect:>+.0f})')
print(f'  OLS回归: {ols_vip:>9.0f} (偏误 {ols_vip-true_vip_effect:>+.0f})')
print(f'  PSM匹配: {psm_vip:>9.0f} (偏误 {psm_vip-true_vip_effect:>+.0f})')
print()
print('注意：所有方法都有偏误——因为loyalty(忠诚度)不可观测。')
print('PSM和OLS的偏误差不多，这里OLS甚至略好——PSM不一定总比OLS好！')
print('什么时候PSM优于OLS？当混淆变量和结果的关系是非线性时，PSM的非参数特性有优势。')

## 📊 区块4：业务场景应用

### 场景1：运营发了一批券——券到底有没有用

**背景**：运营上个月给3000个"高价值用户"发了满200减30的优惠券。活动结束后，数据显示领券用户这个月的平均消费是850元，未领券用户是620元。运营说"券带来了37%的消费增长"。

**你的分析**：

1. **选择偏差自查**：这3000人是定向发的——他们是"过去3个月月均消费>800元的用户"。这不就是"本来就买得多的人"吗？直接对比他们和"所有没领券的人"的平均值=苹果比橙子。

2. **PSM匹配**：
   - 倾向得分模型：用历史消费、活跃度、注册时长、设备类型等估计"被发券的概率"
   - 对每个领券用户，找一个"倾向得分接近但没领券"的用户
   - 匹配后：领券组人均消费850元，匹配的未领券组人均消费790元
   - 券的增量效应 = 850 - 790 = 60元/人

3. **券成本**：30元/张
4. **券ROI**：60元增量 - 30元成本 = 30元利润

5. **结论**：券有效果（60元增量），但不是运营说的230元（850-620）。刨掉选择偏差后，真实增量是60元。发券在当前的券面额下有正ROI。

### 场景2：新用户引导流程改版——评估改版效果

**背景**：产品改了新用户的引导流程（增加了3个步骤的引导动画），想知道改版后的新用户7日留存是否提升了。但改版是全量上线的——没有AB实验。

**PSM方案**：

1. **处理组**：改版后注册的用户
2. **对照组**：改版前1个月注册的用户（未经历新版引导）
3. **协变量**：注册渠道、设备类型、下载来源、注册时间(控制季节性)
4. **PSM匹配**：匹配上述协变量→得到"改版前和改版后特征相似"的两组用户
5. **结果**：PSM估计新版引导提升了7日留存3个百分点(从28%到31%)

**可能的问题（需要在报告中说明）**：
- 时间混淆：改版前后的市场环境可能不同（如竞品动作、节假日）
- 如果是这个担忧→不用PSM，改用DID（如果有同期未改版的对照组市场）

### 场景3：为什么这个PSM做错了——常见误用诊断

**案例A："我把处理后的变量也放进倾向得分模型了"**
- 错在哪：处理后的变量可能是中介变量→控制了中介=低估了处理效应
- 正确做法：倾向得分模型只能用处理前的变量

**案例B："我对匹配后的样本直接跑了OLS t检验"**
- 错在哪：匹配过程引入了不确定性→普通标准误低估了真实的不确定性
- 正确做法：使用Abadie-Imbens稳健标准误，或做bootstrap

**案例C："PSM结果显示SMD都<0.1了，所以我证明VIP有效果"**
- 错在哪：平衡性检验只验证了"可观测变量匹配好了"，不能验证"没有未观测混淆"
- 正确说法："在可观测变量条件下，VIP的估计效应为X。但如果有未观测的重要混淆变量，该估计可能有偏"

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 能说出PSM的四个步骤
- [ ] 能用logistic回归+最近邻匹配实现一个基础PSM
- [ ] 能解释倾向得分的含义（"被处理的概率"）
- [ ] 能区分ATT和ATE

**答案提示**：
- 估计ps→匹配→诊断→估计效应
- sklearn LogisticRegression + 距离匹配
- ps = P(T=1 | X)，在给定X下被处理的概率
- ATT = E[Y(1)-Y(0)|T=1]（处理组的平均效应），ATE = E[Y(1)-Y(0)]（全体的平均效应）

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能解释为什么PSM只控制了可观测混淆（对比DID能控制不可观测的不变混淆）
- [ ] 能说出至少3种匹配方法及其适用场景
- [ ] 能解释卡尺匹配中caliper=0.2×SD(ps)的理由
- [ ] 能解释共同支撑(Common Support)假设

**答案提示**：
- PSM依赖倾向得分模型包含的X→遗漏不可观测混淆→偏误；DID用固定效应消除不随时间变化的未观测混淆
- 最近邻/卡尺/半径/核/分层匹配（各有偏误-方差权衡）
- Austin(2011)通过模拟研究推荐0.2×SD能消除约98%的ps偏差
- 共同支撑：对任何X，0 < P(T=1|X) < 1→处理组和对照组倾向得分有重叠→有可比性

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能设计一个PSM+DID的混合分析方案
- [ ] 能判断给定的业务问题是否适合用PSM
- [ ] 能解释为什么PSM有时比OLS差（以及什么时候更好）
- [ ] 能做Rosenbaum敏感性分析

**答案提示**：
- 先用处理前PSM匹配→在匹配样本上做DID→结合两者优势
- PSM适用条件：有丰富的处理前协变量+共同支撑成立+相信可忽略性假设
- PSM优于OLS：当混淆变量和结果是非线性关系时(PSM非参数)；OLS优于PSM：当线性模型近似正确时(OLS更有效)
- Rosenbaum分析回答："一个未观测混淆要多强才能推翻我的结论？"

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | 周一因果推断基础(反事实/混淆/选择偏差) | PSM通过倾向得分匹配来消除选择偏差 |
| **← 前置依赖** | 逻辑回归(W3) | 倾向得分模型的核心是logistic回归 |
| **→ 后续用途** | 周五PSM平衡性检验与敏感性分析 | 匹配后的诊断和稳健性验证 |
| **→ 后续用途** | PSM+DID组合 | W10-W12综合案例中的混合策略 |
| **业务连接** | 用户分群效果评估 | 如VIP/会员/高价值标签的因果效应 |
| **业务连接** | 渠道归因中的选择偏差处理 | 不同渠道用户质量比较需要PSM |

### 横向连接：与其他分析方法的关联

**与DID的连接**：
- PSM+DID = 先用PSM确保处理前可比→再用DID消除固定效应→双重保障
- DID有平行趋势假设，PSM有可忽略性假设→哪个假设更可信就倾向哪个方法

**与机器学习的连接**：
- 倾向得分模型可以用任何二分类器（RF/GBDT/NN）
- 因果森林(Causal Forest)是PSM的"森林版本"
- Double ML：用ML估计倾向得分和结果模型，然后用正交化估计因果效应

**与实验设计的连接**：
- 分层随机化 = 实验版的"分层PSM"——在实验设计中就保证了平衡
- CUPED(方差缩减)的思想和PSM+回归调整是相通的

## 📝 区块7：当日小结

> **PSM的本质是"降维匹配"——用一个数字（倾向得分）来概括多维协变量，然后匹配这个数字。它的四个步骤是：估计倾向得分→匹配→诊断→估计效应。PSM比朴素对比强在控制了可观测混淆，比DID弱在不能控制不可观测的不变混淆（DID用固定效应做到了这一点）。PSM最大的软肋是"可忽略性假设"不可检验——如果有重要的混淆变量你没测量到，PSM仍然有偏。"PSM+DID"是增强版策略：先用PSM在做处理前匹配（让两组更可比），然后在匹配样本上做DID（消除不随时间变化的混淆）。记住一句话：PSM的成功取决于你测量到的协变量是否"充分"——而这个"充分性"本身是没法从数据中检验的。**

**三条金句带回家**：
1. 倾向得分 = 一个把多维协变量压缩为一维标量的降维工具——它让你在1D上匹配，而不是在p维空间里碰运气
2. PSM只控制了可观测的混淆——这是它和AB实验/RCT最本质的区别
3. PSM后必须做平衡性检验——SMD<0.1是"匹配成功"的最低门槛

### 🔖 课后拓展

1. **练习**：在生成的数据上，对比"Logistic倾向得分"和"GBDT倾向得分"的匹配效果差异

2. **思考**：如果处理组有1000人，对照组有100000人，你会怎么设计匹配策略？（提示：有放回/不放回？卡尺大小？）

3. **预习**：明天要学的PSM平衡性检验——怎么用SMD、QQ图、共同支撑图来判断匹配得好不好

4. **面试准备**：
   - "PSM的四个步骤是什么？"
   - "为什么倾向得分能降维？为什么不是直接在X上匹配？"
   - "PSM最大的局限是什么？什么情况下PSM比OLS更差？"

---

*03analyst_thinking · Week 9 Day 4 · 2026-05-21*